# Agent Workflow Checklist

This notebook turns the repository contract into a short, executable checklist for agents and contributors.


## 1. Load The Shared Workflow Scaffold

### 1.1 Reuse Tested Notebook Helpers

Notebook cells should orchestrate validations and logging, not own those rules inline.


In [1]:
from pathlib import Path

from repo_rag_lab.notebook_scaffolding import build_agent_workflow_context
from repo_rag_lab.notebook_support import (
    assert_minimum_pass_rate,
    assert_no_validation_issues,
    resolve_repo_root,
    write_notebook_run_log,
)
from repo_rag_lab.utilities import utility_summary
from repo_rag_lab.workflow import ask_repository

root = resolve_repo_root(Path.cwd().resolve())
context = build_agent_workflow_context(root)
print(utility_summary(root))


INFO repo_rag_lab.notebook_scaffolding Built agent workflow context with 3 benchmarks and 1 MCP candidates.


Repository utility surfaces:
- make utility-summary / uv run repo-rag utility-summary: list the supported entrypoints
- make ask / uv run repo-rag ask: answer repository-grounded questions
- make discover-mcp / uv run repo-rag discover-mcp: inspect repo-local MCP candidates
- make azure-manifest / uv run repo-rag azure-manifest: write Azure deployment metadata
- make smoke-test / uv run repo-rag smoke-test: validate the core workflow surfaces
- make verify-surfaces / uv run repo-rag verify-surfaces: validate notebooks and Makefile verification surfaces
- root: /home/standard/dspy_rag_in_repo_docs_and_impl1


## 2. Validate Repository Question Answering

### 2.1 Start With A Known Question

Use a repository-specific question before trying open-ended prompts so regressions stay obvious.


In [2]:
baseline = ask_repository("What does this repository research?", root=root)
print(baseline.answer)


Question: What does this repository research?

Evidence:
- /home/standard/dspy_rag_in_repo_docs_and_impl1/data/questions/repository.yaml: - question: What does this repository research? answer_contains: repository - question: Where are inspired implementation summaries stored? answer_contains: documentation/inspired - question: How should agents inspect repository utilities? 
- /home/standard/dspy_rag_in_repo_docs_and_impl1/.github/workflows/ci.yml: ml --all-targets -- -D warnings - name: Build Rust wrapper run: cargo build --manifest-path rust-cli/Cargo.toml - name: Run Rust wrapper against packaged workflow run: >- cargo run --manifest-path rust-cli/Cargo.toml -- ask --question "What
- /home/standard/dspy_rag_in_repo_docs_and_impl1/tests/features/repository_rag.feature: Feature: Repository RAG Scenario: Answer a repository scope question Given the repository root When I ask the repository question "What does this repository research?" Then the answer mentions "repository" Scenario: D

## 3. Run Notebook Assertions

### 3.1 Validate Samples, MCP Discovery, And Benchmarks

The scaffold returns validation issues, MCP candidates, and benchmark results in one payload.


In [3]:
training_issues = context["training_validation_issues"]
population_issues = context["population_validation_issues"]
assert_no_validation_issues(training_issues, context="training samples")
assert_no_validation_issues(population_issues, context="population candidates")
assert_minimum_pass_rate(context["benchmark_summary"], minimum_pass_rate=1.0)
{
    "mcp_candidates": context["mcp_candidates"],
    "benchmark_summary": context["benchmark_summary"],
}


{'mcp_candidates': ['pyproject.toml'],
 'benchmark_summary': {'case_count': 3,
  'pass_count': 3,
  'pass_rate': 1.0,
  'retrieved_source_hits': {'AGENTS.md': 1,
   'README.md': 2,
   'TODO.MD': 1,
   'documentation/inspired/dspy-rag-tutorial.md': 1,
   'documentation/inspired/implementing-rag-with-dspy-technical-guide.md': 2,
   'src/repo_rag_lab/notebook_scaffolding.py': 1,
   'src/repo_rag_lab/training_samples.py': 1,
   'src/repo_rag_lab/utilities.py': 1},
  'matched_source_hits': {'AGENTS.md': 1,
   'README.md': 2,
   'documentation/inspired/implementing-rag-with-dspy-technical-guide.md': 1,
   'src/repo_rag_lab/utilities.py': 1},
  'results': [{'question': 'What does this repository research?',
    'expected_sources': ['README.md', 'src/repo_rag_lab/utilities.py'],
    'retrieved_sources': ['src/repo_rag_lab/utilities.py',
     'src/repo_rag_lab/notebook_scaffolding.py',
     'README.md'],
    'matched_sources': ['src/repo_rag_lab/utilities.py', 'README.md'],
    'passed': True,


## 4. Write A Notebook Run Log

### 4.1 Persist Scaffold Output For Later Review

Notebook logs land in `artifacts/notebook_logs/` so they do not mix with GitHub Actions logs in `samples/logs/`.


In [4]:
log_path = write_notebook_run_log(root, "02-agent-workflow-checklist", context)
print(log_path.relative_to(root))


artifacts/notebook_logs/20260318T001709Z-02-agent-workflow-checklist.json
